In [1]:
import sys
sys.path.insert(0,'/home/ujjwalsrao/project/kaggle-whale/python/')
%load_ext autoreload

In [2]:
%autoreload
from processing.metric.data import data_loader, score_loader, pretrain_loader
from modeling.metric.model import ResNet, Accuracy, CenterLoss
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from utility.optimizer import AdamW
from utility.schedular import CosineLR
from utility.checkpoint import save_model, load_model
from modeling.metric.train import train_model
from modeling.metric.score import score_model

## Training + Validation

- ### 224x224 - finetune

In [3]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 224, 64, False)

In [4]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
# cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-2)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = None
save_path = '../../model/metric/experiment-1/'
epochs = 5
batch = 64
alpha = 0.5

In [5]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [6]:
train_model(*arguments)

100% 9691/9728 [00:56<00:00, 172.88it/s, trn_ac=8.81, trn_ls=115.82, val_ac=22.55, val_ls=6.60]
100% 9691/9728 [00:55<00:00, 173.60it/s, trn_ac=29.08, trn_ls=75.77, val_ac=46.93, val_ls=4.30]
100% 9691/9728 [00:55<00:00, 176.19it/s, trn_ac=48.41, trn_ls=51.60, val_ac=76.86, val_ls=2.54]
100% 9691/9728 [00:55<00:00, 175.32it/s, trn_ac=69.70, trn_ls=35.86, val_ac=84.15, val_ls=1.92]
100% 9691/9728 [00:55<00:00, 173.75it/s, trn_ac=83.02, trn_ls=25.50, val_ac=95.28, val_ls=1.07]


In [7]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [8]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64)

Train Images: 12766 Valid Images: 2931


In [9]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-2)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-1/'
save_path = '../../model/metric/experiment-1/'
epochs = 10
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [10]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [11]:
train_model(*arguments)

Model Loaded: Loss: 1.0728


100% 12766/12800 [00:51<00:00, 247.26it/s, trn_ac=22.31, trn_ls=53.70, val_ac=12.47, val_ls=7.35]
100% 12766/12800 [00:52<00:00, 244.41it/s, trn_ac=38.48, trn_ls=39.36, val_ac=19.04, val_ls=6.59]
100% 12766/12800 [00:51<00:00, 248.56it/s, trn_ac=50.19, trn_ls=29.40, val_ac=26.95, val_ls=5.80]
100% 12766/12800 [00:50<00:00, 250.84it/s, trn_ac=61.11, trn_ls=22.25, val_ac=34.23, val_ls=5.12]
100% 12766/12800 [00:51<00:00, 246.62it/s, trn_ac=74.93, trn_ls=16.83, val_ac=45.29, val_ls=4.37]
100% 12766/12800 [00:51<00:00, 247.56it/s, trn_ac=87.06, trn_ls=12.88, val_ac=48.47, val_ls=4.12]
100% 12766/12800 [00:51<00:00, 247.98it/s, trn_ac=96.54, trn_ls=9.72, val_ac=53.20, val_ls=3.80]
100% 12766/12800 [00:52<00:00, 244.36it/s, trn_ac=99.21, trn_ls=7.48, val_ac=58.89, val_ls=3.50]
100% 12766/12800 [00:51<00:00, 247.54it/s, trn_ac=99.79, trn_ls=5.82, val_ac=60.73, val_ls=3.41]
100% 12766/12800 [00:51<00:00, 248.95it/s, trn_ac=99.95, trn_ls=4.58, val_ac=61.46, val_ls=3.37]


In [12]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 224x224 - retrain

In [13]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 224, 64, False)

In [14]:
name = 'freeze_1_size_224'
model = ResNet(freeze=3).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-3)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-1/'
save_path = '../../model/metric/experiment-2/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [15]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [16]:
train_model(*arguments)

Model Loaded: Loss: 3.3655


100% 9691/9728 [01:12<00:00, 197.01it/s, trn_ac=82.94, trn_ls=3.12, val_ac=94.46, val_ls=1.00]
100% 9691/9728 [01:10<00:00, 197.04it/s, trn_ac=93.43, trn_ls=1.98, val_ac=97.83, val_ls=0.54]
100% 9691/9728 [01:10<00:00, 197.67it/s, trn_ac=97.22, trn_ls=1.48, val_ac=99.23, val_ls=0.25]


In [17]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [18]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64)

Train Images: 12766 Valid Images: 2931


In [19]:
name = 'freeze_1_size_224'
model = ResNet(freeze=3).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-3)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-2/'
save_path = '../../model/metric/experiment-2/'
epochs = 20
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [20]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [21]:
train_model(*arguments)

Model Loaded: Loss: 0.2487


100% 12766/12800 [01:12<00:00, 199.01it/s, trn_ac=70.69, trn_ls=6.21, val_ac=52.39, val_ls=4.43]
100% 12766/12800 [01:12<00:00, 199.14it/s, trn_ac=81.56, trn_ls=5.19, val_ac=56.31, val_ls=4.01]
100% 12766/12800 [01:12<00:00, 199.65it/s, trn_ac=91.25, trn_ls=4.37, val_ac=64.99, val_ls=3.51]
100% 12766/12800 [01:13<00:00, 197.92it/s, trn_ac=97.59, trn_ls=3.74, val_ac=69.70, val_ls=3.18]
100% 12766/12800 [01:12<00:00, 199.72it/s, trn_ac=99.62, trn_ls=3.31, val_ac=72.67, val_ls=2.95]
100% 12766/12800 [01:12<00:00, 195.24it/s, trn_ac=99.91, trn_ls=3.02, val_ac=74.67, val_ls=2.84]
100% 12766/12800 [01:12<00:00, 196.80it/s, trn_ac=99.98, trn_ls=2.80, val_ac=75.52, val_ls=2.79]
100% 12766/12800 [01:12<00:00, 198.95it/s, trn_ac=100.00, trn_ls=2.65, val_ac=77.04, val_ls=2.73]
100% 12766/12800 [01:12<00:00, 198.36it/s, trn_ac=99.99, trn_ls=2.53, val_ac=77.44, val_ls=2.68]
100% 12766/12800 [01:12<00:00, 197.81it/s, trn_ac=100.00, trn_ls=2.42, val_ac=77.09, val_ls=2.64]
100% 12766/12800 [01:12<00:0

In [22]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 448x448 - finetune

In [7]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 448, 64, False)

In [8]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(1e-2)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-2)/4)
load_path = '../../model/metric/experiment-2/'
save_path = '../../model/metric/experiment-3/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [9]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [10]:
train_model(*arguments)

Model Loaded: Loss: 2.4597


100% 9691/9728 [02:45<00:00, 103.16it/s, trn_ac=60.45, trn_ls=3.71, val_ac=83.42, val_ls=1.73]
100% 9691/9728 [02:45<00:00, 101.92it/s, trn_ac=84.35, trn_ls=1.60, val_ac=97.02, val_ls=0.71]
100% 9691/9728 [02:45<00:00, 103.50it/s, trn_ac=94.65, trn_ls=0.73, val_ac=99.80, val_ls=0.25]


In [11]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [3]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64)

Train Images: 12766 Valid Images: 2931


In [4]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(1e-2)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-3/'
save_path = '../../model/metric/experiment-3/'
epochs = 20
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [5]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [6]:
train_model(*arguments)

Model Loaded: Loss: 0.2512


100% 12766/12800 [02:28<00:00, 102.50it/s, trn_ac=57.56, trn_ls=4.38, val_ac=55.63, val_ls=4.24]
100% 12766/12800 [02:28<00:00, 103.43it/s, trn_ac=73.91, trn_ls=2.77, val_ac=64.69, val_ls=3.47]
100% 12766/12800 [02:28<00:00, 103.68it/s, trn_ac=86.63, trn_ls=1.64, val_ac=72.58, val_ls=2.80]
100% 12766/12800 [02:27<00:00, 103.94it/s, trn_ac=96.31, trn_ls=0.87, val_ac=77.49, val_ls=2.44]
100% 12766/12800 [02:29<00:00, 103.94it/s, trn_ac=99.40, trn_ls=0.50, val_ac=81.23, val_ls=2.25]
100% 12766/12800 [02:29<00:00, 103.72it/s, trn_ac=99.90, trn_ls=0.36, val_ac=82.43, val_ls=2.24]
100% 12766/12800 [02:28<00:00, 103.66it/s, trn_ac=99.99, trn_ls=0.28, val_ac=82.64, val_ls=2.25]
100% 12766/12800 [02:27<00:00, 102.92it/s, trn_ac=99.99, trn_ls=0.25, val_ac=83.38, val_ls=2.18]
100% 12766/12800 [02:29<00:00, 103.86it/s, trn_ac=100.00, trn_ls=0.22, val_ac=83.20, val_ls=2.18]
100% 12766/12800 [02:28<00:00, 103.95it/s, trn_ac=100.00, trn_ls=0.20, val_ac=83.46, val_ls=2.19]
100% 12766/12800 [02:28<00:0

In [7]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 448x448 - upsample

In [8]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 448, 64, True)

In [9]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(5e-3)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(5e-3)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-3/'
save_path = '../../model/metric/experiment-4/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [10]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [11]:
train_model(*arguments)

Model Loaded: Loss: 2.1049


100% 12766/12800 [02:26<00:00, 104.58it/s, trn_ac=85.26, trn_ls=2.03, val_ac=59.11, val_ls=3.98]
100% 12766/12800 [02:27<00:00, 104.53it/s, trn_ac=92.35, trn_ls=1.39, val_ac=68.17, val_ls=3.42]
100% 12766/12800 [02:26<00:00, 104.72it/s, trn_ac=98.30, trn_ls=0.74, val_ac=72.45, val_ls=2.96]


In [12]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [13]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, True, True)

Train Images: 12766 Valid Images: 2931


In [14]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(5e-3)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(5e-3)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-4/'
save_path = '../../model/metric/experiment-4/'
epochs = 10
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [15]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [16]:
train_model(*arguments)

Model Loaded: Loss: 2.9629


100% 12766/12800 [02:27<00:00, 104.38it/s, trn_ac=91.47, trn_ls=2.13, val_ac=61.75, val_ls=4.05]
100% 12766/12800 [02:26<00:00, 103.77it/s, trn_ac=94.59, trn_ls=1.14, val_ac=70.84, val_ls=3.37]
100% 12766/12800 [02:27<00:00, 103.95it/s, trn_ac=98.27, trn_ls=0.53, val_ac=77.28, val_ls=2.88]
100% 12766/12800 [02:26<00:00, 104.64it/s, trn_ac=99.48, trn_ls=0.28, val_ac=79.51, val_ls=2.67]
100% 12766/12800 [02:26<00:00, 104.57it/s, trn_ac=99.82, trn_ls=0.19, val_ac=81.11, val_ls=2.54]
100% 12766/12800 [02:27<00:00, 104.40it/s, trn_ac=99.98, trn_ls=0.14, val_ac=81.54, val_ls=2.53]
100% 12766/12800 [02:26<00:00, 104.43it/s, trn_ac=100.00, trn_ls=0.12, val_ac=81.88, val_ls=2.48]
100% 12766/12800 [02:27<00:00, 104.57it/s, trn_ac=99.98, trn_ls=0.11, val_ac=82.24, val_ls=2.46]
100% 12766/12800 [02:27<00:00, 102.68it/s, trn_ac=100.00, trn_ls=0.10, val_ac=82.51, val_ls=2.45]
100% 12766/12800 [02:26<00:00, 104.35it/s, trn_ac=100.00, trn_ls=0.10, val_ac=82.95, val_ls=2.43]


## Training + Scoring

- ### 224x224 - finetune

In [17]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 224, 64, False)

In [18]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
# cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-2)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = None
save_path = '../../model/metric/experiment-5/'
epochs = 5
batch = 64
alpha = 0.5

In [19]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [20]:
train_model(*arguments)

100% 9691/9728 [00:54<00:00, 177.98it/s, trn_ac=9.26, trn_ls=115.32, val_ac=24.10, val_ls=5.89]
100% 9691/9728 [00:55<00:00, 174.37it/s, trn_ac=28.54, trn_ls=75.78, val_ac=45.38, val_ls=4.36]
100% 9691/9728 [00:56<00:00, 171.64it/s, trn_ac=46.83, trn_ls=51.76, val_ac=74.35, val_ls=2.65]
100% 9691/9728 [00:54<00:00, 176.85it/s, trn_ac=67.38, trn_ls=36.07, val_ac=78.42, val_ls=2.49]
100% 9691/9728 [00:55<00:00, 174.54it/s, trn_ac=80.56, trn_ls=25.77, val_ac=93.07, val_ls=1.19]


In [21]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [22]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64, False)

Train Images: 15697 Valid Images: 2931


In [23]:
name = 'freeze_1_size_224'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-2, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-2)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-5/'
save_path = '../../model/metric/experiment-5/'
epochs = 10
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [24]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [25]:
train_model(*arguments)

Model Loaded: Loss: 1.1861


100% 15697/15744 [01:00<00:00, 259.68it/s, trn_ac=19.18, trn_ls=54.08, val_ac=18.63, val_ls=6.61]
100% 15697/15744 [01:00<00:00, 261.13it/s, trn_ac=37.19, trn_ls=36.89, val_ac=45.73, val_ls=4.41]
100% 15697/15744 [01:00<00:00, 259.73it/s, trn_ac=50.18, trn_ls=25.93, val_ac=73.73, val_ls=2.85]
100% 15697/15744 [01:00<00:00, 259.20it/s, trn_ac=64.48, trn_ls=18.52, val_ac=90.88, val_ls=1.74]
100% 15697/15744 [01:00<00:00, 259.63it/s, trn_ac=78.83, trn_ls=13.24, val_ac=97.74, val_ls=0.94]
100% 15697/15744 [01:00<00:00, 260.90it/s, trn_ac=91.17, trn_ls=9.47, val_ac=99.73, val_ls=0.58]
100% 15697/15744 [01:00<00:00, 258.10it/s, trn_ac=97.58, trn_ls=6.81, val_ac=99.93, val_ls=0.31]
100% 15697/15744 [01:00<00:00, 260.37it/s, trn_ac=99.40, trn_ls=4.99, val_ac=99.97, val_ls=0.21]
100% 15697/15744 [01:00<00:00, 258.79it/s, trn_ac=99.87, trn_ls=3.72, val_ac=100.00, val_ls=0.17]
100% 15697/15744 [01:00<00:00, 259.19it/s, trn_ac=99.97, trn_ls=2.83, val_ac=100.00, val_ls=0.13]


In [26]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 224x224 - retrain

In [27]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 224, 64, False)

In [28]:
name = 'freeze_1_size_224'
model = ResNet(freeze=3).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-3)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-5/'
save_path = '../../model/metric/experiment-6/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [29]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [30]:
train_model(*arguments)

Model Loaded: Loss: 0.134


100% 9691/9728 [01:11<00:00, 195.08it/s, trn_ac=90.60, trn_ls=2.06, val_ac=98.74, val_ls=0.62]
100% 9691/9728 [01:12<00:00, 198.21it/s, trn_ac=97.88, trn_ls=1.11, val_ac=99.69, val_ls=0.30]
100% 9691/9728 [01:12<00:00, 197.64it/s, trn_ac=99.36, trn_ls=0.78, val_ac=99.97, val_ls=0.13]


In [31]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [32]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 224, 64, False)

Train Images: 15697 Valid Images: 2931


In [33]:
name = 'freeze_1_size_224'
model = ResNet(freeze=3).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=1e-3, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=1e-3)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=1e-4)
load_path = '../../model/metric/experiment-6/'
save_path = '../../model/metric/experiment-6/'
epochs = 20
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [34]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [35]:
train_model(*arguments)

Model Loaded: Loss: 0.1287


100% 15697/15744 [01:26<00:00, 180.66it/s, trn_ac=71.62, trn_ls=4.79, val_ac=90.61, val_ls=1.92]
100% 15697/15744 [01:26<00:00, 180.62it/s, trn_ac=84.89, trn_ls=3.58, val_ac=99.38, val_ls=0.88]
100% 15697/15744 [01:27<00:00, 179.07it/s, trn_ac=92.63, trn_ls=2.80, val_ac=99.97, val_ls=0.32]
100% 15697/15744 [01:26<00:00, 181.38it/s, trn_ac=97.43, trn_ls=2.28, val_ac=100.00, val_ls=0.17]
100% 15697/15744 [01:26<00:00, 181.29it/s, trn_ac=99.42, trn_ls=1.95, val_ac=100.00, val_ls=0.13]
100% 15697/15744 [01:27<00:00, 179.98it/s, trn_ac=99.91, trn_ls=1.74, val_ac=100.00, val_ls=0.10]
100% 15697/15744 [01:27<00:00, 180.20it/s, trn_ac=99.99, trn_ls=1.59, val_ac=100.00, val_ls=0.08]
100% 15697/15744 [01:26<00:00, 180.74it/s, trn_ac=100.00, trn_ls=1.48, val_ac=100.00, val_ls=0.08]
100% 15697/15744 [01:27<00:00, 179.73it/s, trn_ac=100.00, trn_ls=1.40, val_ac=100.00, val_ls=0.07]
100% 15697/15744 [01:27<00:00, 178.40it/s, trn_ac=100.00, trn_ls=1.32, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [01

In [36]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 448x448 - finetune

In [37]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 448, 64, False)

In [38]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(1e-2)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-6/'
save_path = '../../model/metric/experiment-7/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [39]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [pretrain]
arguments += [pretrain]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [40]:
train_model(*arguments)

Model Loaded: Loss: 0.0313


100% 9691/9728 [02:44<00:00, 104.09it/s, trn_ac=69.01, trn_ls=3.09, val_ac=91.78, val_ls=1.13]
100% 9691/9728 [02:44<00:00, 104.18it/s, trn_ac=90.72, trn_ls=1.18, val_ac=98.26, val_ls=0.33]
100% 9691/9728 [02:45<00:00, 103.88it/s, trn_ac=97.21, trn_ls=0.58, val_ac=99.84, val_ls=0.10]


In [41]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [42]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, False)

Train Images: 15697 Valid Images: 2931


In [43]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(1e-2)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-7/'
save_path = '../../model/metric/experiment-7/'
epochs = 12
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [44]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [45]:
train_model(*arguments)

Model Loaded: Loss: 0.0981


100% 15697/15744 [02:54<00:00, 101.76it/s, trn_ac=54.60, trn_ls=4.94, val_ac=69.39, val_ls=2.85]
100% 15697/15744 [02:54<00:00, 101.99it/s, trn_ac=78.30, trn_ls=2.85, val_ac=97.88, val_ls=0.65]
100% 15697/15744 [02:55<00:00, 102.23it/s, trn_ac=91.02, trn_ls=1.70, val_ac=99.93, val_ls=0.19]
100% 15697/15744 [02:54<00:00, 102.18it/s, trn_ac=97.59, trn_ls=1.08, val_ac=100.00, val_ls=0.10]
100% 15697/15744 [02:54<00:00, 102.29it/s, trn_ac=99.61, trn_ls=0.77, val_ac=100.00, val_ls=0.07]
100% 15697/15744 [02:54<00:00, 100.82it/s, trn_ac=99.97, trn_ls=0.63, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [02:54<00:00, 102.09it/s, trn_ac=100.00, trn_ls=0.55, val_ac=100.00, val_ls=0.06]
100% 15697/15744 [02:55<00:00, 100.66it/s, trn_ac=100.00, trn_ls=0.49, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:54<00:00, 102.40it/s, trn_ac=100.00, trn_ls=0.44, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [02:54<00:00, 101.96it/s, trn_ac=100.00, trn_ls=0.41, val_ac=100.00, val_ls=0.05]
100% 15697/15744 [0

In [46]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

- ### 448x448 - upsample

In [47]:
pretrain = pretrain_loader('../../data/pretrain/image/', '../../data/pretrain/train.csv', 448, 64, True)

In [48]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(1e-2)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(1e-2)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-7/'
save_path = '../../model/metric/experiment-8/'
epochs = 3
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [49]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [50]:
train_model(*arguments)

Model Loaded: Loss: 0.0445


100% 15697/15744 [02:53<00:00, 102.22it/s, trn_ac=61.61, trn_ls=3.75, val_ac=84.69, val_ls=1.87]
100% 15697/15744 [02:54<00:00, 102.56it/s, trn_ac=82.05, trn_ls=2.01, val_ac=99.42, val_ls=0.40]
100% 15697/15744 [02:54<00:00, 101.42it/s, trn_ac=94.01, trn_ls=1.04, val_ac=100.00, val_ls=0.10]


In [51]:
save_model(1, cent_loss, 1, '../../model/metric/cent_loss.pth')

In [52]:
train, valid = data_loader('../../data/boxed/train/','../../data/raw/train.csv', 448, 64, False, True)

Train Images: 15697 Valid Images: 2931


In [53]:
name = 'freeze_1_size_448'
model = ResNet(freeze=1).float().cuda()
xent_loss = CrossEntropyLoss()
cent_loss = CenterLoss()
cent_loss, _ = load_model(cent_loss, '../../model/metric/cent_loss.pth')
metric_fn = Accuracy()
xent_optim = AdamW(model.parameters(), lr=(5e-3)/4, weight_decay=1)
cent_optim = Adam(cent_loss.parameters(), lr=(5e-3)/4)
schedular = CosineLR(xent_optim, T_max=100, T_mult=0.98, eta_min=(1e-4)/4)
load_path = '../../model/metric/experiment-8/'
save_path = '../../model/metric/experiment-8/'
epochs = 10
batch = 64
alpha = 0.5

Model Loaded: Loss: 1


In [54]:
arguments = []
arguments += [name]
arguments += [model]
arguments += [train]
arguments += [valid]
arguments += [alpha]
arguments += [xent_loss]
arguments += [cent_loss]
arguments += [metric_fn]
arguments += [xent_optim]
arguments += [cent_optim]
arguments += [schedular]
arguments += [save_path]
arguments += [load_path]
arguments += [epochs]
arguments += [batch]

In [55]:
train_model(*arguments)

Model Loaded: Loss: 0.1042


100% 15697/15744 [02:56<00:00, 101.75it/s, trn_ac=90.73, trn_ls=2.17, val_ac=97.66, val_ls=0.72]
100% 15697/15744 [02:54<00:00, 100.61it/s, trn_ac=95.48, trn_ls=1.15, val_ac=99.52, val_ls=0.20]
100% 15697/15744 [02:53<00:00, 102.02it/s, trn_ac=98.90, trn_ls=0.62, val_ac=99.93, val_ls=0.10]
100% 15697/15744 [02:54<00:00, 102.35it/s, trn_ac=99.71, trn_ls=0.43, val_ac=99.93, val_ls=0.06]
100% 15697/15744 [02:54<00:00, 102.20it/s, trn_ac=99.93, trn_ls=0.35, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [02:54<00:00, 101.84it/s, trn_ac=99.97, trn_ls=0.32, val_ac=100.00, val_ls=0.04]
100% 15697/15744 [02:55<00:00, 101.82it/s, trn_ac=99.97, trn_ls=0.29, val_ac=100.00, val_ls=0.03]
100% 15697/15744 [02:54<00:00, 101.85it/s, trn_ac=100.00, trn_ls=0.25, val_ac=100.00, val_ls=0.03]
100% 15697/15744 [02:55<00:00, 101.85it/s, trn_ac=100.00, trn_ls=0.24, val_ac=100.00, val_ls=0.03]
100% 15697/15744 [02:57<00:00, 101.97it/s, trn_ac=100.00, trn_ls=0.23, val_ac=100.00, val_ls=0.03]


- ### scoring

In [56]:
data = score_loader('../../data/boxed/test/', 448, 16, True)

Score Images: 7960


In [57]:
model = ResNet(freeze=False).float().cuda()

In [58]:
score_model(model, data, 16, '../../model/metric/experiment-8/', 5)

Model Loaded: Loss: 0.026


100% 7960/7968 [01:00<00:00, 132.27it/s]


Records: 7960 7960


100% 7960/7968 [01:00<00:00, 131.31it/s]


Records: 15920 15920


...

Records: 23880 23880


100% 7960/7968 [01:00<00:00, 131.36it/s]


Records: 31840 31840


100% 7960/7968 [01:00<00:00, 131.50it/s]


Records: 39800 39800
